# Early-Layer-SAE-Feature-Forecasting: Step 3 - activation cache extraction

Builds the ~1.89 GB float16 activation cache that Step 4+ probes train on.

**Inputs:** Gemma-2-2B (gated, license required), GemmaScope layer-20 residual SAE, Pile-10k (HF dataset).

**Outputs (Google Drive at `safety-sae-cache/v1/`):**
- `resid_layer_5.npy`, `resid_layer_8.npy`, `resid_layer_12.npy`, `resid_layer_20.npy`: post-block residual streams, shape `(102400, 2304)`, float16, ~471 MB each
- `feature_acts.npy`: SAE activations for the 5 target features at layer 20, shape `(102400, 5)`, float16
- `token_ids.npy`: token ids, shape `(102400,)`, int32 (row-major: `seq_idx = token_idx // 256`)
- `metadata.json`: config, versions, empirical per-feature fire rates, dataset commit, timestamp

**Runtime:** T4 GPU. End-to-end ~5 min after warm-up (model + SAE load dominate).

**Re-runs are safe:** existing output files are overwritten; the extraction cell clears any pre-existing hooks first.

**After running:** download `safety-sae-cache/v1/` to local `data/cache/v1/`, then `python scripts/check_activation_cache.py`.

## 1. Install dependencies (~1-2 min)

In [ ]:
!pip install -q sae_lens accelerate datasets

## 2. Mount Google Drive

Cache is written to `safety-sae-cache/v1/` under your Drive root so it survives Colab disconnects. Needs ~2 GB free.

In [ ]:
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive'
free_gb = shutil.disk_usage(DRIVE_ROOT).free / 1e9
print(f'Drive free space: {free_gb:.2f} GB')
if free_gb < 2.5:
    raise RuntimeError(f'Need ~2.5 GB free; only {free_gb:.2f} GB available. Free space and re-run.')

## 3. Authenticate with Hugging Face

Gemma-2-2B is gated. Before continuing:
1. Accept the license at https://huggingface.co/google/gemma-2-2b
2. Generate a read token at https://huggingface.co/settings/tokens

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

## 4. Config (single source of truth)

Downstream cells read these constants; edit nothing else.

In [ ]:
CORPUS = 'NeelNanda/pile-10k'              # web text training distribution
N_SEQUENCES = 400                          # → 400 × 256 = 102,400 tokens
SEQ_LEN = 256                              # BOS-prefixed
EARLY_LAYERS = [5, 8, 12]                  # precursor probes
LATE_LAYER = 20                            # SAE layer (upper-bound probe + label source)
ALL_LAYERS = EARLY_LAYERS + [LATE_LAYER]   # = [5, 8, 12, 20]
FEATURE_INDICES = [9989, 817, 12730, 892, 1031]
FEATURE_THEMES = {9989: 'refusal', 817: 'deception', 12730: 'ethics', 892: 'sycophancy-adj', 1031: 'harm'}
SAE_RELEASE = 'gemma-scope-2b-pt-res-canonical'
SAE_ID = 'layer_20/width_16k/canonical'
BATCH_SIZE = 8
OUTPUT_DIR = '/content/drive/MyDrive/safety-sae-cache/v1'
MODEL_NAME = 'google/gemma-2-2b'

# Derived
N_TOKENS = N_SEQUENCES * SEQ_LEN
assert N_SEQUENCES % BATCH_SIZE == 0, 'N_SEQUENCES must be divisible by BATCH_SIZE'
N_BATCHES = N_SEQUENCES // BATCH_SIZE

print(f'Will cache {N_TOKENS:,} tokens × {len(ALL_LAYERS)} layers × 2304 dim (fp16) = '
      f'{N_TOKENS * len(ALL_LAYERS) * 2304 * 2 / 1e9:.2f} GB residuals + '
      f'{N_TOKENS * len(FEATURE_INDICES) * 2 / 1e6:.2f} MB features over {N_BATCHES} batches')

## 5. Load Gemma-2-2B

Same pattern as the smoke test: HF `transformers` with `device_map=device` + `bfloat16` + `low_cpu_mem_usage=True`. Avoids the host-RAM OOM that `transformer_lens` triggers on Colab free.

In [ ]:
import torch, gc
from transformers import AutoTokenizer, AutoModelForCausalLM

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cpu':
    raise RuntimeError('No GPU detected. Runtime → Change runtime type → T4 GPU.')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map=device,
    low_cpu_mem_usage=True,
)
model.eval()
gc.collect(); torch.cuda.empty_cache()

d_model = model.config.hidden_size
n_layers = model.config.num_hidden_layers
assert max(ALL_LAYERS) < n_layers, f'requested layer {max(ALL_LAYERS)} >= n_layers={n_layers}'
print(f'Loaded {MODEL_NAME}: n_layers={n_layers}, d_model={d_model}, '
      f'GPU mem={torch.cuda.memory_allocated()/1e9:.2f} GB')

## 6. Load the GemmaScope SAE (layer 20, width 16k, canonical)

In [ ]:
from sae_lens import SAE

sae, cfg_dict, _ = SAE.from_pretrained(release=SAE_RELEASE, sae_id=SAE_ID, device=device)
sae.eval()
hook_layer = cfg_dict.get('hook_layer')
if hook_layer is not None and hook_layer != LATE_LAYER:
    raise ValueError(f'SAE hook_layer={hook_layer} != LATE_LAYER={LATE_LAYER}')
assert sae.cfg.d_in == d_model, f'SAE d_in={sae.cfg.d_in} != model d_model={d_model}'
print(f'SAE loaded: d_in={sae.cfg.d_in}, d_sae={sae.cfg.d_sae}, hook_layer={hook_layer}, '
      f'W_enc.dtype={sae.W_enc.dtype}')

## 7. Load and tokenize Pile-10k

Stream documents in dataset order, tokenize without special tokens, concatenate to one flat buffer of `400 × 255 = 102,000` raw tokens, reshape to `(400, 255)`, prepend a BOS column → `(400, 256)`. Cross-document spans inside a sequence are acceptable (matches GemmaScope's training packing).

We pin the dataset revision so the cache is reproducible across re-runs.

In [ ]:
import numpy as np
from datasets import load_dataset
from huggingface_hub import HfApi

try:
    DATASET_COMMIT = HfApi().dataset_info(CORPUS).sha
    print(f'Pinning {CORPUS} to commit {DATASET_COMMIT[:12]}')
except Exception as e:
    DATASET_COMMIT = None
    print(f'(could not resolve commit for {CORPUS}: {e}; using default revision)')

ds = load_dataset(CORPUS, split='train', revision=DATASET_COMMIT)
print(f'{len(ds):,} documents in {CORPUS}')

raw_tokens_needed = N_SEQUENCES * (SEQ_LEN - 1)  # leave one slot per seq for BOS
buffer, docs_used = [], 0
for doc in ds:
    ids = tokenizer(doc['text'], add_special_tokens=False)['input_ids']
    buffer.extend(ids)
    docs_used += 1
    if len(buffer) >= raw_tokens_needed:
        break
assert len(buffer) >= raw_tokens_needed, (
    f'Corpus exhausted: only {len(buffer):,} tokens from {docs_used} docs, need {raw_tokens_needed:,}'
)
buffer = buffer[:raw_tokens_needed]

arr = np.array(buffer, dtype=np.int32).reshape(N_SEQUENCES, SEQ_LEN - 1)
bos_col = np.full((N_SEQUENCES, 1), tokenizer.bos_token_id, dtype=np.int32)
input_ids_all = np.concatenate([bos_col, arr], axis=1)
assert input_ids_all.shape == (N_SEQUENCES, SEQ_LEN)
print(f'Tokenized {docs_used} docs → buffer of {len(buffer):,} tokens → input_ids {input_ids_all.shape}')

## 8. Pre-allocate output arrays in host RAM

~1.89 GB total. Allocating up front (rather than appending per batch) keeps memory deterministic and avoids realloc.

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

resid_arrays = {L: np.zeros((N_TOKENS, d_model), dtype=np.float16) for L in ALL_LAYERS}
feature_arr = np.zeros((N_TOKENS, len(FEATURE_INDICES)), dtype=np.float16)
token_ids_arr = input_ids_all.reshape(-1).astype(np.int32)

total_gb = (sum(a.nbytes for a in resid_arrays.values()) + feature_arr.nbytes + token_ids_arr.nbytes) / 1e9
print(f'Pre-allocated {total_gb:.2f} GB '
      f'({len(ALL_LAYERS)} resid layers × {resid_arrays[ALL_LAYERS[0]].nbytes/1e6:.0f} MB '
      f'+ features {feature_arr.nbytes/1e6:.2f} MB '
      f'+ token_ids {token_ids_arr.nbytes/1e6:.2f} MB)')

## 9. Register hooks and run the forward+encode loop

Per batch: forward Gemma-2-2B, capture post-block residuals at layers `[5, 8, 12, 20]` via `register_forward_hook` on `model.model.layers[L]`, encode the layer-20 residual through the SAE, index out the 5 target feature columns, write into the pre-allocated arrays.

50 batches at `BATCH_SIZE=8`. ~3-4 min on T4.

The cell starts by clearing any pre-existing forward hooks on those layers so re-running this cell during a session does not double-register.

In [ ]:
import torch
from tqdm.auto import tqdm

# Idempotency: drop any pre-existing forward hooks on the model's transformer blocks
for layer_mod in model.model.layers:
    layer_mod._forward_hooks.clear()

_batch_cache = {}
def _make_hook(L):
    def hook(module, inp, out):
        h = out[0] if isinstance(out, tuple) else out
        _batch_cache[L] = h.detach()
    return hook

hooks = [model.model.layers[L].register_forward_hook(_make_hook(L)) for L in ALL_LAYERS]
feature_idx_t = torch.tensor(FEATURE_INDICES, device=device, dtype=torch.long)

try:
    input_ids_t = torch.from_numpy(input_ids_all).to(device)
    for b in tqdm(range(N_BATCHES), desc='extract'):
        s, e = b * BATCH_SIZE, (b + 1) * BATCH_SIZE
        batch = input_ids_t[s:e]
        _batch_cache.clear()
        with torch.no_grad():
            _ = model(batch)

        tok_s, tok_e = s * SEQ_LEN, e * SEQ_LEN

        resid_late = _batch_cache[LATE_LAYER].to(sae.W_enc.dtype)
        with torch.no_grad():
            feats = sae.encode(resid_late)[:, :, feature_idx_t]  # (B, T, 5)
        feature_arr[tok_s:tok_e] = feats.reshape(-1, len(FEATURE_INDICES)).to(torch.float16).cpu().numpy()

        for L in ALL_LAYERS:
            h = _batch_cache[L].reshape(-1, d_model).to(torch.float16).cpu().numpy()
            resid_arrays[L][tok_s:tok_e] = h
finally:
    for h in hooks:
        h.remove()

print(f'Extracted {N_TOKENS:,} tokens × {len(ALL_LAYERS)} layers; '
      f'GPU mem={torch.cuda.memory_allocated()/1e9:.2f} GB')

## 10. Save artifacts + metadata to Drive

Uncompressed `.npy` files (compressed `.npz` does not mmap, which Step 4 needs).

In [ ]:
import json, datetime, platform
from pathlib import Path
import sae_lens, transformers, datasets as _datasets

out = Path(OUTPUT_DIR)
out.mkdir(parents=True, exist_ok=True)

for L in ALL_LAYERS:
    np.save(out / f'resid_layer_{L}.npy', resid_arrays[L])
np.save(out / 'feature_acts.npy', feature_arr)
np.save(out / 'token_ids.npy', token_ids_arr)

fire_rates = {str(idx): float((feature_arr[:, i] > 0).mean())
              for i, idx in enumerate(FEATURE_INDICES)}

metadata = {
    'created_at': datetime.datetime.utcnow().isoformat() + 'Z',
    'platform': platform.platform(),
    'config': {
        'model': MODEL_NAME,
        'corpus': CORPUS,
        'dataset_commit': DATASET_COMMIT,
        'n_sequences': N_SEQUENCES,
        'seq_len': SEQ_LEN,
        'n_tokens': N_TOKENS,
        'early_layers': EARLY_LAYERS,
        'late_layer': LATE_LAYER,
        'all_layers': ALL_LAYERS,
        'feature_indices': FEATURE_INDICES,
        'feature_themes': {str(k): v for k, v in FEATURE_THEMES.items()},
        'sae_release': SAE_RELEASE,
        'sae_id': SAE_ID,
        'batch_size': BATCH_SIZE,
        'd_model': d_model,
        'dtype_resid': 'float16',
        'dtype_features': 'float16',
        'dtype_token_ids': 'int32',
        'layout': 'flat row-major; seq_idx = token_idx // seq_len',
    },
    'empirical': {
        'fire_rates': fire_rates,
    },
    'versions': {
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'sae_lens': sae_lens.__version__,
        'datasets': _datasets.__version__,
        'numpy': np.__version__,
    },
}

(out / 'metadata.json').write_text(json.dumps(metadata, indent=2))

print(f'Saved to {out}/')
for p in sorted(out.iterdir()):
    print(f'  {p.name:<22} {p.stat().st_size / 1e6:>10.2f} MB')

## 11. Sanity statistics

Three checks:
1. **Empirical per-feature fire rates** vs Neuronpedia's reported rates. Some difference is expected vs Neuronpedia and we evaluate on different corpora, but order-of-magnitude divergence would mean something is wrong.
2. **Top-activating token contexts** per feature (should look semantically coherent with the auto-interp label).
3. **Per-layer residual stats**, in particular max-abs as a float16-clipping check (fp16 ceiling is ±65504; GemmaScope SAE max acts are ~125 so the residuals should be well below clipping).

In [ ]:
NEURONPEDIA_FRAC_NONZERO = {9989: 0.00746, 817: 0.00763, 12730: 0.00485, 892: 0.00279, 1031: 0.00481}

print('1. Per-feature empirical fire rate (Pile-10k, N=102,400 tokens):')
print(f'   {"idx":>6}  {"theme":<16}  {"empirical":>10}  {"neuronpedia":>12}  {"ratio":>7}')
for i, idx in enumerate(FEATURE_INDICES):
    emp = float((feature_arr[:, i] > 0).mean())
    npd = NEURONPEDIA_FRAC_NONZERO[idx]
    ratio = emp / npd if npd > 0 else float('inf')
    print(f'   {idx:>6}  {FEATURE_THEMES[idx]:<16}  {emp:>10.5f}  {npd:>12.5f}  {ratio:>6.2f}x')

print()
print('2. Top-3 activating contexts per feature (5 tokens before, 1 after):')
for i, idx in enumerate(FEATURE_INDICES):
    acts = feature_arr[:, i].astype(np.float32)
    print(f'\n   Feature {idx} ({FEATURE_THEMES[idx]}):')
    top_pos = np.argsort(acts)[::-1][:3]
    if acts[top_pos[0]] == 0:
        print('     [no firings on this corpus]')
        continue
    for p in top_pos:
        if acts[p] == 0:
            break
        ctx_s = max(0, int(p) - 5)
        ctx_e = min(N_TOKENS, int(p) + 2)
        ctx = tokenizer.decode(token_ids_arr[ctx_s:ctx_e].tolist())
        target = tokenizer.decode([int(token_ids_arr[p])])
        print(f'     act={acts[p]:>6.2f}  pos={int(p):>6}  target={target!r:<14}  ctx={ctx!r}')

print()
print('3. Per-layer residual stats:')
print(f'   {"layer":>5}  {"|x|_max (full)":>16}  {"|x|_mean (samp)":>17}  {"L2 (samp)":>11}')
rng = np.random.default_rng(0)
sample_idx = rng.choice(N_TOKENS, size=8192, replace=False)
for L in ALL_LAYERS:
    x = resid_arrays[L]
    max_abs = float(np.abs(x).max())
    samp = x[sample_idx].astype(np.float32)
    mean_abs = float(np.abs(samp).mean())
    l2_mean = float(np.linalg.norm(samp, axis=1).mean())
    flag = '  WARN: near fp16 ceiling' if max_abs > 6e4 else ''
    print(f'   {L:>5}  {max_abs:>16.2f}  {mean_abs:>17.4f}  {l2_mean:>11.2f}{flag}')

## 12. Download to local

Cache lives in your Drive at `safety-sae-cache/v1/`. To use it for Step 4:

1. Open https://drive.google.com
2. Right-click `safety-sae-cache/v1` → **Download** (Drive zips and serves it).
3. Unzip into the repo so the layout is:
   ```
   data/cache/v1/resid_layer_5.npy
   data/cache/v1/resid_layer_8.npy
   data/cache/v1/resid_layer_12.npy
   data/cache/v1/resid_layer_20.npy
   data/cache/v1/feature_acts.npy
   data/cache/v1/token_ids.npy
   data/cache/v1/metadata.json
   ```
4. Verify: `python scripts/check_activation_cache.py`

The download is ~1.89 GB; on a typical home connection, 5–15 min.

## Done

Cache is in Drive. After downloading and verifying locally, you're ready for Step 4 (probe training).